In [60]:
import numpy as np

In [61]:
%%capture
%run ./DTW.ipynb

In [2]:
def midi_measure(measure_boundaries, wp):
    """
    Get the midi measure boundaries as frames
    """

    midi_measure_frames = []
    start_midi = 0

    for i in range(measure_boundaries.shape[0]):
        end_xml = measure_boundaries[i]
        adjustment = 0

        while (np.where(wp[1] == end_xml)[0]).size < 1:
            end_xml -= 1 # adjust boundary slightly if exact frame is not listed
            adjustment += 1

        end_idx = np.where(wp[1] == end_xml)[0][0]

        end_midi = wp[0][end_idx] + adjustment
        midi_measure_frames.append((start_midi,end_midi))
        start_midi = end_midi

    return midi_measure_frames

In [63]:
def midi_times(mid_frames, midi_file, save = False):
    """convert frames of midi to time using fs"""

    midi_data = np.load(midi_file)
    fs = midi_data['fs']
    midi_measure_times = mid_frames/fs

    file_name = midi_file[14:-8] + ".txt"

    if save:
        np.savetxt(file_name, midi_measure_times[:,0], delimiter = ' ')
    return midi_measure_times

In [64]:
def compare_measures(midi_file, xml_file, xml_measure_frames, midi_measure_frames):
    """
    Compare alignment frame by frame to see how many notes match up
    """

    F1data = np.load(midi_file) # 88 x N
    F1 = F1data['roll']
    F2data = np.load(xml_file) # 88 x M
    F2 = F2data['roll']

    lxml = len(xml_measure_frames)
    lmidi = len(midi_measure_frames)
    soft_accuracy = []
    count = 0

    for i in range (max(lxml,lmidi)-1):

        xml_frame = F2[:,int(xml_measure_frames[i])]
        midi_frame = F1[:,midi_measure_frames[i][1]]

        # check accuracy with dot_product
        actual = np.dot(midi_frame, xml_frame)
        expected = np.dot(xml_frame, xml_frame)
        soft_accuracy.append(actual/expected)

        if (actual == expected):
            count += 1

    strict_accuracy = 100* count/(lxml-1)
    soft_accuracy = np.mean(soft_accuracy)*100
    print(f"Soft Accuracy: {soft_accuracy}")
    print(f"Strict Accuracy: {strict_accuracy}")


In [65]:
def compare_alignment(midi_file, xml_file, wp):
    """
    Compare alignment frame by frame to see how many notes match up
    """

    F1data = np.load(midi_file) # 88 x N
    F1 = F1data['roll']
    F2data = np.load(xml_file) # 88 x M
    F2 = F2data['roll']
    
    accuracy = []
    count = 0

    for i in range(wp.shape[1]):
        xml_frame = F2[:,wp[1,i]]
        midi_frame = F1[:,wp[0,i]]


        actual = np.dot(midi_frame, xml_frame)
        expected = np.dot(xml_frame, xml_frame)
        if (actual == expected):
            count += 1

    accuracy = 100* count/wp.shape[1]
    print(f"Alignment Accuracy: {accuracy}")

In [66]:
featfile1 = 'midi_features/schumann_arabeske_park.mid.npz'
featfile2 = 'xml_features/schumann_arabeske.musicxml.npz'

In [67]:
steps = np.array([1,1,1,2,2,1,1,3,3,1,1,4,4,1]).reshape((-1,2))
weights = np.array([2,2,2,2,2,2,2])
downsample = 1
wp, C = alignDTW(featfile1, featfile2, steps, weights, downsample)

(88, 7435)
(88, 7608)


In [68]:
xml_data = np.load(featfile2)
measure_boundaries = xml_data['measure_boundaries']
midi_data = np.load(featfile1)
fs = midi_data['fs']

In [69]:
midi_measure_frames = midi_measure(measure_boundaries, wp)
times = midi_times(midi_measure_frames, featfile1)

In [70]:
compare_measures(featfile1, featfile2, measure_boundaries, midi_measure_frames)
compare_alignment(featfile1, featfile2, wp)

Soft Accuracy: 94.20289855072464
Strict Accuracy: 86.95652173913044
Alignment Accuracy: 81.8746470920384
